# Claims Text-to-SQL — Fine-TuningQLoRA fine-tune of a small instruct model on the CMS-style claims adjudication schema,using Unsloth + TRL SFT. Runs on a free Colab T4.**Before running:** upload `data/train.jsonl` (produced by `python prepare_dataset.py`)and `prompt.py` + `schema.py` to the Colab session, or clone the repo into Colab.The training text is built by `prompt.build_training_text`, the same function`inference.py` calls at serve time. That shared call is deliberate: in the inventoryversion of this project, training used a plain `instruction\noutput` format whileinference sent a chat template with few-shot examples, so the model was asked atserve time for a format it had never been trained to produce. Keep the import.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"!pip install -q --no-deps trl peft accelerate bitsandbytes datasets huggingface_hub

In [ ]:
import json, osfrom datasets import Datasetfrom unsloth import FastLanguageModelfrom trl import SFTTrainer, SFTConfigfrom huggingface_hub import login# Prefer a Colab secret or environment variable over a pasted literal.login(token=os.environ["HF_TOKEN"])

## 1. Load the dataset`prepare_dataset.py` has already validated every pair by executing it against thewarehouse, and has already written the `text` field using the shared prompt builder.Nothing is reformatted here — reformatting is where train/serve skew gets introduced.

In [ ]:
TRAIN_PATH = "data/train.jsonl"records = [json.loads(line) for line in open(TRAIN_PATH, encoding="utf-8") if line.strip()]dataset = Dataset.from_list([{"text": r["text"]} for r in records])print(f"{len(dataset)} training examples")print("-" * 70)print(dataset[0]["text"][:1200])

## 2. Load the base modelAny small instruct checkpoint works. Qwen2.5-1.5B-Instruct fits comfortably in a T4at 4-bit and is a fair size for the claim being tested: that a small local model,fine-tuned on one schema's business definitions, beats a much larger model that hasto infer those definitions from the prompt.Whatever you pick, set `BASE_MODEL` to the same id when running the eval, so thebaseline is the same checkpoint without the fine-tune rather than a different model.

In [ ]:
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"MAX_SEQ_LENGTH = 2048model, tokenizer = FastLanguageModel.from_pretrained(    model_name=BASE_MODEL,    max_seq_length=MAX_SEQ_LENGTH,    load_in_4bit=True,)

In [ ]:
model = FastLanguageModel.get_peft_model(    model,    r=16,    lora_alpha=16,    target_modules=[        "q_proj", "k_proj", "v_proj", "o_proj",        "gate_proj", "up_proj", "down_proj",    ],    lora_dropout=0,    bias="none",    use_gradient_checkpointing="unsloth",    random_state=42,)

## 3. TrainThree epochs rather than two: the dataset is small (~158 pairs) and the hand-writtenbusiness-rule pairs are the minority of it, so the definitions need more than acouple of passes to stick. Watch the loss — if it flattens early, drop back to two.

In [ ]:
trainer = SFTTrainer(    model=model,    tokenizer=tokenizer,    train_dataset=dataset,    args=SFTConfig(        output_dir="outputs",        per_device_train_batch_size=2,        gradient_accumulation_steps=4,        num_train_epochs=3,        learning_rate=2e-4,        warmup_ratio=0.05,        lr_scheduler_type="linear",        logging_steps=5,        fp16=True,        bf16=False,        optim="adamw_8bit",        seed=42,        dataset_text_field="text",        max_seq_length=MAX_SEQ_LENGTH,        report_to="none",    ),)trainer.train()

## 4. Smoke test before pushingGenerate against a held-out question the model has never seen. If the output is notrecognisable SQL, something is wrong with the prompt format — fix it here ratherthan discovering it after the push.

In [ ]:
from prompt import build_prompt, extract_sqlFastLanguageModel.for_inference(model)question = "What percentage of our claims pay on the first pass?"inputs = tokenizer([build_prompt(question)], return_tensors="pt").to("cuda")out = model.generate(**inputs, max_new_tokens=256, do_sample=False)completion = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)print(extract_sql(completion))

## 5. Push the merged modelMerged 16-bit so `inference.py` can load it with plain Transformers on CPU, noUnsloth or GPU needed at serve time.

In [ ]:
REPO_ID = "your-username/claims-text-to-sql"model.push_to_hub_merged(    REPO_ID,    tokenizer=tokenizer,    token=os.environ["HF_TOKEN"],    save_method="merged_16bit",)print(f"Pushed to https://huggingface.co/{REPO_ID}")

## 6. Then run the eval locally```bashexport FINETUNED_MODEL=your-username/claims-text-to-sqlexport BASE_MODEL=Qwen/Qwen2.5-1.5B-Instructpython eval/run_eval.py --backends gold finetuned base --save````gold` must come back at 100%. If it doesn't, the harness is broken and no othernumber in the run means anything.Put the resulting table in the README. If the fine-tune does not beat the baseline,say so there — a negative result honestly reported is worth more than a positive onethat a reader cannot reproduce.